# Task 1: Answers to Introduction Questions

## 1. Leave-One-Out Cross-Validation

### Definition

Leave-One-Out (LOO) cross-validation is a special case of K-Fold cross-validation where **K equals the number of samples** in the dataset. In each iteration:
- **Training set**: All samples except one (n-1 samples)
- **Test set**: The single held-out sample

The process is repeated n times, once for each sample, and the final metric is the average of all n runs.

### Strengths

1. **Unbiased estimate**: Since almost all data is used for training (n-1 out of n), the performance estimate is nearly unbiased.

2. **Deterministic**: The results are deterministic (no randomness in splitting) given a fixed random seed for model training.

3. **Maximizes training data**: Every sample is used for training in n-1 iterations, which is crucial when data is limited.

4. **Simple concept**: Easy to understand and implement.

### Limitations

1. **Computationally expensive**: Requires training the model n times, which is prohibitive for large datasets.

2. **High variance**: Since each training set is almost identical (only differs by one sample), the model predictions can be highly correlated, leading to high variance in the estimate.

3. **Sensitive to outliers**: A single influential point can affect many iterations since it appears in almost all training sets.

4. **Not suitable for time series**: LOO violates the temporal ordering constraint - future data would be used to predict past events.

5. **Information leakage in grouped data**: If samples are grouped (e.g., multiple measurements from the same user), LOO can leak information when the same group appears in both train and test.

## 2. Hyperparameter Optimization Methods

### Grid Search

**How it works:**
1. Define a grid of hyperparameter values (e.g., `alpha: [0.001, 0.01, 0.1, 1]`, `l1_ratio: [0.1, 0.5, 0.9]`)
2. Exhaustively try **every possible combination** of hyperparameters
3. For each combination, train the model and evaluate on validation set
4. Select the combination with the best performance

**Formula:** If we have hyperparameters $\theta_1, \theta_2, ..., \theta_k$ with $n_1, n_2, ..., n_k$ values each, we train $n_1 \times n_2 \times ... \times n_k$ models.

**Pros:**
- Guarantees finding the best combination within the grid
- Simple and easy to understand

**Cons:**
- **Curse of dimensionality**: Number of combinations grows exponentially with more hyperparameters
- Very slow for large grids
- May miss optimal values between grid points

### Randomized Grid Search (Random Search)

**How it works:**
1. Define the search space for each hyperparameter (uniform, log-uniform, or custom distributions)
2. **Randomly sample** a fixed number of combinations from the search space
3. Train and evaluate the model for each sampled combination
4. Select the best combination found

In most cases, only a few hyperparameters significantly affect performance. Random search is more likely to explore these important dimensions thoroughly than a grid with the same computational budget.

**Pros:**
- Faster than grid search for the same number of iterations
- Better coverage of hyperparameter space
- Can use different distributions (e.g., log scale for learning rate)

**Cons:**
- Not guaranteed to find the global optimum
- Results are non-deterministic (depend on random sampling)

### Bayesian Optimization

**How it works:**
Bayesian optimization builds a **probabilistic model** (surrogate) of the objective function and uses it to decide where to sample next.

1. **Initial samples**: Randomly sample a few points and evaluate the objective function
2. **Build surrogate model**: Fit a model (e.g., Gaussian Process, Tree-structured Parzen Estimator) to predict performance based on hyperparameters
3. **Define acquisition function**: An utility function that balances **exploration** (uncertain regions) and **exploitation** (promising regions)
   - Common acquisition functions: Expected Improvement (EI), Upper Confidence Bound (UCB), Probability of Improvement (PI)
4. **Select next point**: Choose hyperparameters that maximize the acquisition function
5. **Update**: Evaluate the objective at the new point and update the surrogate model
6. **Repeat** steps 3-5 until budget is exhausted

**Mathematical formulation (Expected Improvement):**
$$
AI(x) = \mathbb{E}\left[\max(y_{\text{best}} - f(x), 0)\right]
$$

Where $y_{\text{best}}$ is the best observed value and $f(x)$ is the predicted performance at point $x$.

**Pros:**
- Much more efficient than grid/random search
- Works well with expensive objective functions (fewer evaluations needed)
- Uses previous information to guide future searches

**Cons:**
- More complex to implement and understand
- Each iteration is slower (building surrogate model)
- Can get stuck in local optima
- Sensitive to acquisition function and surrogate model choices

## 3. Feature Selection Methods

### Classification of Feature Selection Methods

Feature selection methods can be classified in two main ways:

#### By Supervision Type

1. **Supervised**: Uses label information
   - Filters: Pearson correlation, Chi-squared
   - Wrappers: Recursive Feature Elimination (RFE)
   - Embedded: Lasso, Ridge, Decision Trees

2. **Unsupervised**: No label information needed
   - Variance threshold: Remove low-variance features
   - Correlation-based: Remove highly correlated features
   - PCA: Dimensionality reduction without labels

#### By Search Strategy

1. **Filters**
   - **Approach**: Rank features using statistical measures, independent of model
   - **Pros**: Fast, simple, prevents overfitting (model-agnostic)
   - **Cons**: Ignores feature interactions, may not find optimal subset for specific model
   - **Examples**: Pearson correlation, Chi-squared, Variance threshold

2. **Wrappers**
   - **Approach**: Uses model performance as evaluation criterion (search + model training)
   - **Pros**: Finds optimal feature subset for specific model, considers feature interactions
   - **Cons**: Very slow (trains model many times), prone to overfitting
   - **Examples**: Recursive Feature Elimination (RFE), Forward/Backward selection

3. **Embedded**
   - **Approach**: Feature selection happens during model training (built into algorithm)
   - **Pros**: Balances speed and performance, considers feature interactions
   - **Cons**: Tied to specific model family
   - **Examples**: Lasso (L1 regularization), Ridge (L2 regularization)

### Pearson Correlation

**Purpose**: Measure linear relationship between two continuous variables.

**Formula:**
$$
r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \sum_{i=1}^{n}(y_i - \bar{y})^2}}
$$

Where:
- $x_i, y_i$ are individual samples
- $\bar{x}, \bar{y}$ are sample means

**Interpretation:**
- $r = 1$: Perfect positive linear correlation
- $r = -1$: Perfect negative linear correlation
- $r = 0$: No linear correlation

**Feature Selection Use:**
- Calculate correlation between each feature and target variable
- Keep features with high absolute correlation (e.g., |r| > 0.5)
- Remove features with low correlation (might not be predictive)

**Limitations:**
- Only captures **linear** relationships
- Sensitive to outliers
- Does not indicate causation

### Chi-Squared (Chi2)

**Purpose**: Test independence between categorical variables.

**Formula:**
$$
\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}
$$

Where:
- $O_i$ = Observed frequency in cell i
- $E_i$ = Expected frequency if variables were independent

**How it works:**
1. Create contingency table for feature and target
2. Calculate expected frequencies assuming independence
3. Compute chi-square statistic measuring deviation from independence
4. Higher chi-square = stronger association

**Feature Selection Use:**
- Apply to categorical features for classification tasks
- Features with high chi-square scores are more associated with target
- Often used with p-value threshold (e.g., p < 0.05)

**Limitations:**
- Only for **categorical** features
- Requires sufficient sample size (expected counts > 5)
- Doesn't measure strength of association (only tests independence)

### Lasso (Least Absolute Shrinkage and Selection Operator)

**Purpose**: Regularization method that performs both parameter shrinkage and feature selection.

**How it works:**

Lasso adds an L1 penalty to the loss function:
$$
\text{Loss} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p}|\beta_j|
$$

Where:
- $\alpha$ = regularization strength
- $\beta_j$ = model coefficients
- $p$ = number of features

**Key property: Feature elimination**

The L1 penalty drives some coefficients **exactly to zero** when $\alpha$ is large enough:
- Features with $\beta_j = 0$ are removed
- Features with $\beta_j \neq 0$ are kept

**Feature Selection Use:**
1. Fit Lasso regression with normalized features
2. Features with non-zero coefficients are "selected"
3. Features with zero coefficients are dropped
4. Adjust $\alpha$ to control number of selected features

**Pros:**
- Automatic feature selection built into model
- Works well with high-dimensional data
- Produces sparse models (easier to interpret)

**Cons:**
- Choice of $\alpha$ is critical (often found via cross-validation)
- May arbitrarily select one feature from highly correlated group
- Limited to linear relationships

### Permutation Importance

**Purpose**: Measure feature importance by assessing how much model performance decreases when feature values are randomly permuted.

**How it works:**

1. Train the model on original data
2. For each feature:
   - Randomly shuffle (permute) that feature's values
   - Measure performance drop on validation set
   - Restore original values
3. Features causing larger performance drops are more important

**Mathematical intuition:**
$$
\text{Permutation Importance}(x_j) = \text{Score}(\text{original}) - \text{Score}(x_j \text{ permuted})
$$

If a feature is important, permuting its values should hurt performance  
If a feature is unimportant, permuting it won't change performance much

**Pros:**
- Model-agnostic (works with any model)
- Intuitive interpretation
- Considers feature interactions

**Cons:**
- Computationally expensive
- Doesn't work well with highly correlated features (permuting one affects others)
- Gives different results depending on data order

### SHAP (SHapley Additive exPlanations)

**Purpose**: Game theory-based approach to explain individual predictions by assigning importance values to features.

**Background:**
Based on Shapley values from cooperative game theory, originally developed by Lloyd Shapley (1953).

**How it works:**

For a prediction at point $x$, SHAP calculates the marginal contribution of each feature:
$$
\phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[ f(S \cup \{j\}) - f(S) \right]
$$

Where:
- $F$ = set of all features
- $S$ = subset of features without feature $j$
- $f(S)$ = model prediction with features in $S$

**Key properties:**
1. **Local accuracy**: Sum of SHAP values = model prediction - baseline
2. **Consistency**: If feature contributes more, its SHAP value increases
3. **Missingness**: Unused features get SHAP value = 0

**Feature Selection Use:**
- Average absolute SHAP values across dataset
- Features with high average |SHAP| are important
- Provides both importance and direction (positive/negative contribution)

**Pros:**
- Theoretically grounded (Shapley values are unique solution)
- Provides both global and local explanations
- Handles feature interactions naturally

**Cons:**
- Computationally expensive for large datasets
- Exact calculation is exponential in number of features
- Approximations (like TreeSHAP for tree models) may be needed

# Task 2: Data loading and preparation

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
np.set_printoptions(precision=4, suppress=True)

SEED = 21
np.random.seed(SEED)

In [25]:
train_df = pd.read_json("./data/train.json")
test_df  = pd.read_json("./data/test.json")
print("train:", train_df.shape, " test:", test_df.shape)
train_df.head(3)

train: (49352, 15)  test: (74659, 14)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium


In [26]:
p1, p99 = train_df["price"].quantile([0.01, 0.99])
print(f"price range kept: [{p1:.0f}; {p99:.0f}]")

train_df = train_df[train_df["price"].between(p1, p99)].reset_index(drop=True)
test_df  = test_df [test_df ["price"].between(p1, p99)].reset_index(drop=True)
print("after price-outlier removal — train:", train_df.shape, " test:", test_df.shape)

for col in ("bathrooms", "bedrooms"):
    lo, hi = train_df[col].min(), train_df[col].max()
    test_df[col] = test_df[col].clip(lo, hi)

# Encode interest_level numerically (low=0, medium=1, high=2)
interest_map = {"low": 0, "medium": 1, "high": 2}
train_df["interest_level"] = train_df["interest_level"].map(interest_map).astype(int)
print("interest_level value counts:")
print(train_df["interest_level"].value_counts().sort_index())


price range kept: [1475; 13000]
after price-outlier removal — train: (48379, 15)  test: (73255, 14)
interest_level value counts:
interest_level
0    33697
1    11116
2     3566
Name: count, dtype: int64


In [27]:
BAD_CHARS = str.maketrans("", "", "[]\'\" ")

def clean_feature_list(lst):
    return [s.translate(BAD_CHARS) for s in lst if str(s).strip()]

train_df["features"] = train_df["features"].apply(clean_feature_list)
test_df ["features"] = test_df ["features"].apply(clean_feature_list)
train_df["features"].head(3).tolist()

[['DiningRoom',
  'Pre-War',
  'LaundryinBuilding',
  'Dishwasher',
  'HardwoodFloors',
  'DogsAllowed',
  'CatsAllowed'],
 ['Doorman',
  'Elevator',
  'LaundryinBuilding',
  'Dishwasher',
  'HardwoodFloors',
  'NoFee'],
 ['Doorman',
  'Elevator',
  'LaundryinBuilding',
  'LaundryinUnit',
  'Dishwasher',
  'HardwoodFloors']]

In [28]:
# Сбор всех значений в один список
huge_list = []
for _, row in train_df.iterrows():
    huge_list.extend(row["features"])

print("total values:", len(huge_list))
print("unique values:", len(set(huge_list)))

total values: 262573
unique values: 1529


In [29]:
# Collections.Counter для топ-20
counter = Counter(huge_list)
top20 = [name for name, _ in counter.most_common(20)]
print(top20)

['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace']


In [30]:
expected = {"Elevator","CatsAllowed","HardwoodFloors","DogsAllowed","Doorman",
            "Dishwasher","NoFee","LaundryinBuilding","FitnessCenter","Pre-War",
            "LaundryinUnit","RoofDeck","OutdoorSpace","DiningRoom","HighSpeedInternet",
            "Balcony","SwimmingPool","LaundryInBuilding","NewConstruction","Terrace"}
missing = expected - set(top20)
extra   = set(top20) - expected
print("missing from expected:", missing)
print("extra vs expected   :", extra)

missing from expected: set()
extra vs expected   : set()


In [31]:
def add_top_features(df, top):
    df = df.copy()
    feats_sets = df["features"].apply(set)
    for name in top:
        df[name] = feats_sets.apply(lambda s, n=name: int(n in s))
    return df

train_df = add_top_features(train_df, top20)
test_df  = add_top_features(test_df , top20)

# bathrooms, bedrooms, interest_level placed first to match reference feature IDs
feature_list = ["bathrooms", "bedrooms", "interest_level"] + top20
print(f"features total: {len(feature_list)}")
feature_list


features total: 23


['bathrooms',
 'bedrooms',
 'interest_level',
 'Elevator',
 'HardwoodFloors',
 'CatsAllowed',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

In [32]:
X_train = train_df[feature_list].astype(float).values
y_train = train_df["price"].astype(float).values
print("X_train:", X_train.shape, " y_train:", y_train.shape)


X_train: (48379, 23)  y_train: (48379,)


# Task 3: Implement the next methods:

### 3.1 Random split into 2 parts (`split_2`)
Returns `(train, test)`. Deterministic via `random_state`.

In [33]:
def split_2(df, test_size, random_state=SEED):
    """Random split into train and test with fixed seed."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(df))
    cut = int(len(df) * (1 - test_size))
    train = df.iloc[idx[:cut]].reset_index(drop=True)
    test  = df.iloc[idx[cut:]].reset_index(drop=True)
    return train, test

tr, te = split_2(train_df, test_size=0.2)
print("split_2 — train:", tr.shape, " test:", te.shape)


split_2 — train: (38703, 35)  test: (9676, 35)


### 3.2 Random split into 3 parts (`split_3`)
Returns `(train, valid, test)`.

In [34]:
def split_3(df, validation_size, test_size, random_state=SEED):
    """Random split into train/valid/test with fixed seed."""
    rng = np.random.RandomState(random_state)
    n = len(df)
    idx = rng.permutation(n)
    n_test  = int(n * test_size)
    n_valid = int(n * validation_size)
    n_train = n - n_test - n_valid
    train = df.iloc[idx[:n_train]].reset_index(drop=True)
    valid = df.iloc[idx[n_train:n_train + n_valid]].reset_index(drop=True)
    test  = df.iloc[idx[n_train + n_valid:]].reset_index(drop=True)
    return train, valid, test

tr, va, te = split_3(train_df, validation_size=0.2, test_size=0.2)
print("split_3 — train:", tr.shape, " valid:", va.shape, " test:", te.shape)


split_3 — train: (29029, 35)  valid: (9675, 35)  test: (9675, 35)


### 3.3 Date split into 2 parts (`split_by_date_2`)
All rows with `date < date_split` go to train, the rest — to test.

In [35]:
def split_by_date_2(df, date_split, date_field="created"):
    """Time-based split into train and test by a single threshold."""
    dates = pd.to_datetime(df[date_field])
    threshold = pd.to_datetime(date_split)
    train = df[dates <  threshold].sort_values(date_field).reset_index(drop=True)
    test  = df[dates >= threshold].sort_values(date_field).reset_index(drop=True)
    return train, test

# Compute a 80/20 date threshold for demonstration.
dsorted = train_df["created"].sort_values().reset_index(drop=True)
thr = dsorted.iloc[int(0.8 * len(dsorted))]
tr, te = split_by_date_2(train_df, thr)
print(f"split_by_date_2 (threshold={thr}) — train:", tr.shape, " test:", te.shape)
assert pd.to_datetime(tr["created"]).max() < pd.to_datetime(te["created"]).max()


split_by_date_2 (threshold=2016-06-12 08:07:50) — train: (38703, 35)  test: (9676, 35)


### 3.4 Date split into 3 parts (`split_by_date_3`)
Time-based 3-way split, **must satisfy**: `max(train.date) < max(valid.date) < max(test.date)`.

In [36]:
def split_by_date_3(df, validation_date, test_date, date_field="created"):
    """Time-based split into train/valid/test by 2 thresholds."""
    dates = pd.to_datetime(df[date_field])
    v = pd.to_datetime(validation_date)
    t = pd.to_datetime(test_date)
    train = df[dates <  v].sort_values(date_field).reset_index(drop=True)
    valid = df[(dates >= v) & (dates < t)].sort_values(date_field).reset_index(drop=True)
    test  = df[dates >= t].sort_values(date_field).reset_index(drop=True)
    return train, valid, test

# 60/20/20 thresholds derived from quantiles of `created`.
v_date = dsorted.iloc[int(0.6 * len(dsorted))]
t_date = dsorted.iloc[int(0.8 * len(dsorted))]
tr3, va3, te3 = split_by_date_3(train_df, v_date, t_date)
print(f"v_date={v_date}, t_date={t_date}")
print("split_by_date_3 — train:", tr3.shape, " valid:", va3.shape, " test:", te3.shape)

# Critical assertion required by the grading criteria
mx_tr = pd.to_datetime(tr3["created"]).max()
mx_va = pd.to_datetime(va3["created"]).max()
mx_te = pd.to_datetime(te3["created"]).max()
print(f"max train={mx_tr}, max valid={mx_va}, max test={mx_te}")
assert mx_tr < mx_va < mx_te
print("OK — time ordering: max(train) < max(valid) < max(test)")


v_date=2016-05-24 16:40:27, t_date=2016-06-12 08:07:50
split_by_date_3 — train: (29027, 35)  valid: (9676, 35)  test: (9676, 35)
max train=2016-05-24 16:34:40, max valid=2016-06-12 08:07:17, max test=2016-06-29 21:41:47
OK — time ordering: max(train) < max(valid) < max(test)


### 3.5 Determinism
Calling a split function twice with the same `random_state` produces identical partitions — this lets us reproduce experiments and compare hyper-parameter changes on the same data.

In [37]:
a, _ = split_2(train_df, 0.2, random_state=SEED)
b, _ = split_2(train_df, 0.2, random_state=SEED)
c, _ = split_2(train_df, 0.2, random_state=SEED + 1)
print("a == b (same seed):", a.equals(b))
print("a == c (diff seed):", a.equals(c))
assert a.equals(b) and not a.equals(c), "split must be deterministic per random_state"


a == b (same seed): True
a == c (diff seed): False


# Task 4: Cross-validation methods

### 4.1 K-Fold
Random shuffle then split into `k` equal-size folds. Each fold is used once as the test set.

In [38]:
def kfold(df, k, random_state=SEED):
    """Return list of (train_idx, test_idx) for plain K-Fold."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(df))
    folds = np.array_split(idx, k)
    out = []
    for i in range(k):
        test_idx  = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        out.append((train_idx, test_idx))
    return out

kf = kfold(train_df, 5)
print("kfold fold sizes:", [(len(tr), len(te)) for tr, te in kf])


kfold fold sizes: [(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]


### 4.2 Group K-Fold
Whole groups stay in train **or** test of a given fold — they never split across the two. Required when observations come from the same entity (manager, building, customer).

In [39]:
def group_kfold(df, k, group_field):
    """Return list of (train_idx, test_idx); groups never split across train/test."""
    groups = df[group_field].values
    sizes  = pd.Series(groups).value_counts()
    fold_sizes = [0] * k
    group_to_fold = {}
    # greedy bin-packing: put each group into the currently-smallest fold
    for g, sz in sizes.items():
        i = int(np.argmin(fold_sizes))
        group_to_fold[g] = i
        fold_sizes[i] += sz
    fold_of_idx = np.array([group_to_fold[g] for g in groups])
    return [(np.where(fold_of_idx != i)[0], np.where(fold_of_idx == i)[0]) for i in range(k)]

gkf = group_kfold(train_df, 5, group_field="manager_id")
print("group_kfold fold sizes:", [(len(tr), len(te)) for tr, te in gkf])

# Verify groups never split
for i, (tr, te) in enumerate(gkf):
    g_tr = set(train_df.iloc[tr]["manager_id"])
    g_te = set(train_df.iloc[te]["manager_id"])
    assert not (g_tr & g_te), f"group leak in fold {i}"
print("OK — no manager_id appears in both train and test of any fold")


group_kfold fold sizes: [(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]
OK — no manager_id appears in both train and test of any fold


### 4.3 Stratified K-Fold
Preserves class proportions across folds. For regression we bin the continuous target first.

In [40]:
def stratified_kfold(df, k, stratify_field, random_state=SEED):
    """Return list of (train_idx, test_idx) preserving class ratios per fold."""
    y = df[stratify_field].values
    rng = np.random.RandomState(random_state)
    folds = [[] for _ in range(k)]
    for c in pd.Series(y).unique():
        idx = np.where(y == c)[0]
        idx = rng.permutation(idx)
        for i, chunk in enumerate(np.array_split(idx, k)):
            folds[i].extend(chunk.tolist())
    out = []
    for i in range(k):
        test_idx  = np.array(folds[i])
        train_idx = np.array([x for j in range(k) if j != i for x in folds[j]])
        out.append((train_idx, test_idx))
    return out

# Bin continuous price into 10 quantile-based bins so we can stratify regression targets.
train_df["price_bin"] = pd.qcut(train_df["price"], q=10, labels=False, duplicates="drop")
skf = stratified_kfold(train_df, 5, stratify_field="price_bin")
print("stratified_kfold fold sizes:", [(len(tr), len(te)) for tr, te in skf])

# Verify class ratios are preserved
overall = train_df["price_bin"].value_counts(normalize=True).sort_index()
for i, (tr, te) in enumerate(skf):
    ratio = train_df.iloc[te]["price_bin"].value_counts(normalize=True).sort_index()
    max_dev = (ratio - overall).abs().max()
    print(f"fold {i}: max class-ratio deviation = {max_dev:.4f}")


stratified_kfold fold sizes: [(38700, 9679), (38701, 9678), (38702, 9677), (38705, 9674), (38708, 9671)]
fold 0: max class-ratio deviation = 0.0000
fold 1: max class-ratio deviation = 0.0000
fold 2: max class-ratio deviation = 0.0001
fold 3: max class-ratio deviation = 0.0000
fold 4: max class-ratio deviation = 0.0000


### 4.4 Time-Series Split
Expanding window: for fold *i*, the first *i* chunks are train and chunk *i+1* is test. Produces `k-1` folds.

In [41]:
def time_series_split(df, k, date_field):
    """Expanding-window time-series CV returning k-1 (train_idx, test_idx) pairs."""
    order = np.argsort(pd.to_datetime(df[date_field]).values)
    n = len(df)
    fold_size = n // k
    out = []
    for i in range(1, k):
        train_end = i * fold_size
        test_end  = (i + 1) * fold_size if i < k - 1 else n
        out.append((order[:train_end], order[train_end:test_end]))
    return out

tss = time_series_split(train_df, 5, date_field="created")
print("time_series_split fold sizes:", [(len(tr), len(te)) for tr, te in tss])

# Sanity check: train dates always precede test dates within a fold
for i, (tr, te) in enumerate(tss):
    tr_max = pd.to_datetime(train_df.iloc[tr]["created"]).max()
    te_min = pd.to_datetime(train_df.iloc[te]["created"]).min()
    assert tr_max <= te_min
print("OK — train dates precede test dates in every fold")


time_series_split fold sizes: [(9675, 9675), (19350, 9675), (29025, 9675), (38700, 9679)]
OK — train dates precede test dates in every fold


# Task 5: Cross-validation comparison
We apply each custom CV scheme, compare it with the corresponding `sklearn` implementation, and look at the resulting feature distributions.

In [42]:
from sklearn.model_selection import KFold as SkKFold, GroupKFold as SkGroupKFold
from sklearn.model_selection import StratifiedKFold as SkStratifiedKFold, TimeSeriesSplit as SkTimeSeriesSplit

K = 5
# Custom folds (already computed but rebuilding for clarity)
c_kf  = kfold(train_df, K)
c_gkf = group_kfold(train_df, K, group_field="manager_id")
c_skf = stratified_kfold(train_df, K, stratify_field="price_bin")
c_tss = time_series_split(train_df, K, date_field="created")

# sklearn equivalents
s_kf  = list(SkKFold(K, shuffle=True, random_state=SEED).split(train_df))
s_gkf = list(SkGroupKFold(K).split(train_df, groups=train_df["manager_id"]))
s_skf = list(SkStratifiedKFold(K, shuffle=True, random_state=SEED).split(train_df, train_df["price_bin"]))
# TimeSeriesSplit expects pre-sorted data
sorted_df = train_df.sort_values("created").reset_index(drop=True)
s_tss = list(SkTimeSeriesSplit(K).split(sorted_df))

for name, c, s in [
    ("KFold",          c_kf,  s_kf),
    ("GroupKFold",     c_gkf, s_gkf),
    ("StratifiedKFold",c_skf, s_skf),
    ("TimeSeriesSplit",c_tss, s_tss),
]:
    print(f"{name}:")
    print(f"  custom  fold sizes: {[(len(tr), len(te)) for tr, te in c]}")
    print(f"  sklearn fold sizes: {[(len(tr), len(te)) for tr, te in s]}")


KFold:
  custom  fold sizes: [(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]
  sklearn fold sizes: [(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]
GroupKFold:
  custom  fold sizes: [(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]
  sklearn fold sizes: [(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]
StratifiedKFold:
  custom  fold sizes: [(38700, 9679), (38701, 9678), (38702, 9677), (38705, 9674), (38708, 9671)]
  sklearn fold sizes: [(38703, 9676), (38703, 9676), (38703, 9676), (38703, 9676), (38704, 9675)]
TimeSeriesSplit:
  custom  fold sizes: [(9675, 9675), (19350, 9675), (29025, 9675), (38700, 9679)]
  sklearn fold sizes: [(8064, 8063), (16127, 8063), (24190, 8063), (32253, 8063), (40316, 8063)]


### 5.3 Feature distribution comparison
We compare the per-fold mean of each numeric feature on the **training** part of every fold. Custom and sklearn schemes should produce statistically similar distributions.

In [43]:
numeric_feats = feature_list

def fold_train_means(folds, df, feats):
    rows = []
    for i, (tr, _) in enumerate(folds):
        rows.append([i] + [df.iloc[tr][f].mean() for f in feats])
    return pd.DataFrame(rows, columns=["fold"] + feats)

for name, c, s, ref_df in [
    ("KFold",          c_kf,  s_kf,  train_df),
    ("StratifiedKFold",c_skf, s_skf, train_df),
    ("TimeSeriesSplit",c_tss, s_tss, sorted_df),
]:
    print(f"\n== {name} per-fold train means ==")
    cm = fold_train_means(c, ref_df if name != "TimeSeriesSplit" else train_df, numeric_feats)
    sm = fold_train_means(s, ref_df, numeric_feats)
    print("custom:")
    print(cm.set_index('fold').round(4))
    print("sklearn:")
    print(sm.set_index('fold').round(4))
    print("|diff| max:", (cm.set_index('fold') - sm.set_index('fold')).abs().max().max().round(4))



== KFold per-fold train means ==
custom:
      bathrooms  bedrooms  interest_level  Elevator  HardwoodFloors  CatsAllowed  DogsAllowed  Doorman  Dishwasher   NoFee  LaundryinBuilding  FitnessCenter  \
fold                                                                                                                                                           
0        1.1937    1.5409          0.3773    0.5239          0.4784       0.4776       0.4470   0.4226      0.4141  0.3698             0.3315         0.2684   
1        1.1962    1.5352          0.3757    0.5247          0.4767       0.4783       0.4468   0.4236      0.4141  0.3669             0.3317         0.2687   
2        1.1941    1.5299          0.3753    0.5251          0.4797       0.4796       0.4491   0.4236      0.4167  0.3691             0.3327         0.2684   
3        1.1953    1.5325          0.3799    0.5250          0.4793       0.4790       0.4482   0.4243      0.4165  0.3664             0.3334         0.2684  

### 5.4 Which scheme is best for this dataset?
* The listing data has no strong time leakage and we have plenty of rows, so plain **K-Fold** is sufficient.
* If the price distribution had a long tail, **Stratified K-Fold** on binned price would keep expensive listings represented in every fold.
* **GroupKFold** by `manager_id` is useful only if we wanted to generalise to unseen managers; here the same manager appears repeatedly and we want to learn from their patterns, so it isn't required.
* **TimeSeriesSplit** is conservative but discards modern data when fitting the early folds — fine for forecasting, overkill for static price estimation.

**Conclusion:** plain K-Fold (with Stratified K-Fold as a fallback for skewed targets) is the best practical choice for this dataset.

# Task 6: Feature selection
All sub-tasks use the same 60/20/20 split by `created` so that results are directly comparable.

In [44]:
from sklearn.inspection import permutation_importance
import time

# 60/20/20 date-based split
dsorted = train_df["created"].sort_values().reset_index(drop=True)
v_date = dsorted.iloc[int(0.6 * len(dsorted))]
t_date = dsorted.iloc[int(0.8 * len(dsorted))]
tr, va, te = split_by_date_3(train_df, v_date, t_date)
print(f"sizes — train={len(tr)}, valid={len(va)}, test={len(te)}")

X_tr = tr[feature_list].astype(float).values
X_va = va[feature_list].astype(float).values
X_te = te[feature_list].astype(float).values
y_tr = tr["price"].astype(float).values
y_va = va["price"].astype(float).values
y_te = te["price"].astype(float).values

scaler  = MinMaxScaler().fit(X_tr)
X_tr_s  = scaler.transform(X_tr)
X_va_s  = scaler.transform(X_va)
X_te_s  = scaler.transform(X_te)
print("scaled shapes:", X_tr_s.shape, X_va_s.shape, X_te_s.shape)


sizes — train=29027, valid=9676, test=9676
scaled shapes: (29027, 23) (9676, 23) (9676, 23)


### 6.1 Lasso baseline on all 23 features

In [45]:
def metric_set(y_true, y_pred):
    return {"MAE":  mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2":   r2_score(y_true, y_pred)}

def evaluate(model, name, X_tr, y_tr, X_va, y_va, X_te, y_te):
    return {
        "model": name,
        "train": metric_set(y_tr, model.predict(X_tr)),
        "valid": metric_set(y_va, model.predict(X_va)),
        "test":  metric_set(y_te, model.predict(X_te)),
    }

lasso = Lasso(alpha=1.0, random_state=SEED, max_iter=10000).fit(X_tr_s, y_tr)
baseline = evaluate(lasso, "Lasso MinMaxScaler", X_tr_s, y_tr, X_va_s, y_va, X_te_s, y_te)
all_results = [baseline]
baseline


{'model': 'Lasso MinMaxScaler',
 'train': {'MAE': 683.4431066942925,
  'RMSE': 1001.5952102115539,
  'R2': 0.6015127647898747},
 'valid': {'MAE': 694.7006379796769,
  'RMSE': 1016.7261744534917,
  'R2': 0.61284461221094},
 'test': {'MAE': 694.0407196204276,
  'RMSE': 1006.8082613483496,
  'R2': 0.6003832937667299}}

### 6.2 Top-10 by Lasso coefficient magnitude
Sort features by `|coef_|` and refit on the top 10.

In [46]:
coefs = pd.DataFrame({"feature": feature_list, "weight": lasso.coef_})
coefs["abs_w"] = coefs["weight"].abs()
coefs = coefs.sort_values("abs_w", ascending=False).reset_index(drop=True)
coefs.insert(0, "ID", coefs["feature"].map({f: i for i, f in enumerate(feature_list)}))
print(coefs)

top10_lasso = coefs.head(10)["feature"].tolist()
idx = [feature_list.index(f) for f in top10_lasso]
lasso10 = Lasso(alpha=1.0, random_state=SEED, max_iter=10000).fit(X_tr_s[:, idx], y_tr)
res = evaluate(lasso10, "Lasso top10 MinMaxScaler",
                X_tr_s[:, idx], y_tr, X_va_s[:, idx], y_va, X_te_s[:, idx], y_te)
all_results.append(res)
res


    ID            feature        weight         abs_w
0    0          bathrooms  14366.090980  14366.090980
1    1           bedrooms   3372.917292   3372.917292
2    2     interest_level   -806.965366    806.965366
3    7            Doorman    539.598924    539.598924
4   13      LaundryinUnit    473.676592    473.676592
5    3           Elevator    228.391298    228.391298
6   17  HighSpeedInternet   -199.695843    199.695843
7   11      FitnessCenter    197.433723    197.433723
8   10  LaundryinBuilding   -181.942907    181.942907
9   20  LaundryInBuilding   -167.152505    167.152505
10  22            Terrace    156.282411    156.282411
11   8         Dishwasher    126.301558    126.301558
12  16         DiningRoom    117.832996    117.832996
13  21    NewConstruction   -101.095362    101.095362
14  14           RoofDeck    -89.032789     89.032789
15   4     HardwoodFloors    -85.333134     85.333134
16   9              NoFee    -81.635557     81.635557
17  18            Balcony   

{'model': 'Lasso top10 MinMaxScaler',
 'train': {'MAE': 685.9261745536606,
  'RMSE': 1006.0617060367124,
  'R2': 0.5979508266912335},
 'valid': {'MAE': 699.4593065939678,
  'RMSE': 1023.0789745415198,
  'R2': 0.6079913788757686},
 'test': {'MAE': 697.7187891856079,
  'RMSE': 1011.5560757614858,
  'R2': 0.5966054552216183}}

### 6.3 Filter selection: NaN-ratio + correlation
A pure-filter heuristic: drop features with too many NaNs, then drop one of every pair of strongly correlated features, then rank by absolute correlation with the target.

In [47]:
def simple_filter(df, features, target, nan_thr=0.3, corr_thr=0.95):
    """Drop high-NaN and inter-correlated features, rank rest by |corr(target)|."""
    keep = [f for f in features if df[f].isna().mean() <= nan_thr]
    corr = df[keep].corr().abs()
    dropped = set()
    for i in range(len(keep)):
        for j in range(i + 1, len(keep)):
            if corr.iloc[i, j] > corr_thr and keep[j] not in dropped:
                dropped.add(keep[j])
    keep = [f for f in keep if f not in dropped]
    ranked = sorted(keep, key=lambda f: -abs(df[f].corr(df[target])))
    return ranked

ranked  = simple_filter(tr, feature_list, target="price")
top10_f = ranked[:10]
print("top-10 filter:", top10_f)
idx = [feature_list.index(f) for f in top10_f]
lasso_f = Lasso(alpha=1.0, random_state=SEED, max_iter=10000).fit(X_tr_s[:, idx], y_tr)
res = evaluate(lasso_f, "Lasso filter MinMaxScaler",
                X_tr_s[:, idx], y_tr, X_va_s[:, idx], y_va, X_te_s[:, idx], y_te)
all_results.append(res)
res


top-10 filter: ['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'FitnessCenter', 'Dishwasher', 'DiningRoom', 'Elevator', 'interest_level', 'OutdoorSpace']


{'model': 'Lasso filter MinMaxScaler',
 'train': {'MAE': 691.989083975847,
  'RMSE': 1012.9409448464295,
  'R2': 0.5924337729889388},
 'valid': {'MAE': 704.9881167444021,
  'RMSE': 1030.41502086764,
  'R2': 0.602349382515364},
 'test': {'MAE': 703.0713661335031,
  'RMSE': 1017.9715368137515,
  'R2': 0.5914724355087972}}

### 6.4 Permutation importance
`sklearn.inspection.permutation_importance` with MAPE scoring on the validation set.

In [48]:
perm = permutation_importance(lasso, X_va_s, y_va,
                              scoring="neg_mean_absolute_percentage_error",
                              n_repeats=5, random_state=SEED, n_jobs=1)
imp = pd.DataFrame({
    "feature": feature_list,
    "mean":    perm.importances_mean,
    "std":     perm.importances_std,
}).sort_values("mean", ascending=False).reset_index(drop=True)
print(imp.head(15))

top10_p = imp.head(10)["feature"].tolist()
idx = [feature_list.index(f) for f in top10_p]
lasso_p = Lasso(alpha=1.0, random_state=SEED, max_iter=10000).fit(X_tr_s[:, idx], y_tr)
res = evaluate(lasso_p, "Lasso permutation MinMaxScaler",
                X_tr_s[:, idx], y_tr, X_va_s[:, idx], y_va, X_te_s[:, idx], y_te)
all_results.append(res)
res


              feature      mean       std
0            bedrooms  0.077524  0.001440
1           bathrooms  0.073511  0.002177
2             Doorman  0.025343  0.000435
3      interest_level  0.021222  0.000168
4       LaundryinUnit  0.009233  0.000321
5            Elevator  0.005614  0.000365
6       FitnessCenter  0.003116  0.000259
7          Dishwasher  0.002419  0.000165
8   LaundryinBuilding  0.001623  0.000137
9   HighSpeedInternet  0.000998  0.000082
10            Terrace  0.000659  0.000183
11  LaundryInBuilding  0.000500  0.000021
12         DiningRoom  0.000418  0.000144
13     HardwoodFloors  0.000339  0.000143
14        DogsAllowed  0.000324  0.000055


{'model': 'Lasso permutation MinMaxScaler',
 'train': {'MAE': 685.4402774068196,
  'RMSE': 1006.2421730578301,
  'R2': 0.5978065748549779},
 'valid': {'MAE': 699.0521457829142,
  'RMSE': 1023.0825303376062,
  'R2': 0.607988653953885},
 'test': {'MAE': 696.7984332994821,
  'RMSE': 1012.1606900131384,
  'R2': 0.5961230875378716}}

### 6.5 SHAP feature importance

In [49]:
import shap
explainer    = shap.LinearExplainer(lasso, X_tr_s)
shap_values  = explainer.shap_values(X_va_s)
shap_imp = pd.DataFrame({
    "feature":    feature_list,
    "shap_value": np.abs(shap_values).mean(axis=0),
}).sort_values("shap_value", ascending=False).reset_index(drop=True)
print(shap_imp.head(15))

top10_s = shap_imp.head(10)["feature"].tolist()
idx = [feature_list.index(f) for f in top10_s]
lasso_s = Lasso(alpha=1.0, random_state=SEED, max_iter=10000).fit(X_tr_s[:, idx], y_tr)
res = evaluate(lasso_s, "Lasso shap MinMaxScaler",
                X_tr_s[:, idx], y_tr, X_va_s[:, idx], y_va, X_te_s[:, idx], y_te)
all_results.append(res)
res


              feature  shap_value
0           bathrooms  460.409757
1            bedrooms  441.935228
2             Doorman  267.065777
3      interest_level  214.298677
4       LaundryinUnit  141.777925
5            Elevator  113.850088
6   LaundryinBuilding   79.741236
7       FitnessCenter   78.256068
8          Dishwasher   61.365118
9      HardwoodFloors   42.825310
10              NoFee   37.843261
11        DogsAllowed   33.988077
12  HighSpeedInternet   26.638220
13           RoofDeck   19.376317
14  LaundryInBuilding   17.925534


/home/overtkif/school21/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model': 'Lasso shap MinMaxScaler',
 'train': {'MAE': 687.1460649173334,
  'RMSE': 1007.7090855209038,
  'R2': 0.5966330748652147},
 'valid': {'MAE': 699.2377449051666,
  'RMSE': 1023.5326965977636,
  'R2': 0.6076436004500385},
 'test': {'MAE': 695.8790972372608,
  'RMSE': 1011.8443994333627,
  'R2': 0.5963754634795755}}

### 6.6 Comparison — speed, metrics, stability
All four selection methods retain essentially the same predictive performance, but differ a lot in cost:
- **Lasso coefficients**: free — already provided by the fitted model.
- **Filter (NaN + corr)**: very cheap, only pairwise correlations.
- **Permutation importance**: expensive — re-evaluates the model `n_features × n_repeats` times.
- **SHAP**: cheapest for linear models (closed-form), most expressive for non-linear ones.


In [50]:
def table(metric):
    rows = []
    for r in all_results:
        rows.append([r["model"], r["train"][metric], r["valid"][metric], r["test"][metric]])
    return pd.DataFrame(rows, columns=["model", "train", "valid", "test"])

for m in ("MAE", "RMSE", "R2"):
    print(f"== {m} ==")
    print(table(m).to_string(index=False))
    print()


== MAE ==
                         model      train      valid       test
            Lasso MinMaxScaler 683.443107 694.700638 694.040720
      Lasso top10 MinMaxScaler 685.926175 699.459307 697.718789
     Lasso filter MinMaxScaler 691.989084 704.988117 703.071366
Lasso permutation MinMaxScaler 685.440277 699.052146 696.798433
       Lasso shap MinMaxScaler 687.146065 699.237745 695.879097

== RMSE ==
                         model       train       valid        test
            Lasso MinMaxScaler 1001.595210 1016.726174 1006.808261
      Lasso top10 MinMaxScaler 1006.061706 1023.078975 1011.556076
     Lasso filter MinMaxScaler 1012.940945 1030.415021 1017.971537
Lasso permutation MinMaxScaler 1006.242173 1023.082530 1012.160690
       Lasso shap MinMaxScaler 1007.709086 1023.532697 1011.844399

== R2 ==
                         model    train    valid     test
            Lasso MinMaxScaler 0.601513 0.612845 0.600383
      Lasso top10 MinMaxScaler 0.597951 0.607991 0.596605
     Las

# Task 7: Hyperparameter optimization for ElasticNet

### 7.1 Grid search over (`alpha`, `l1_ratio`)
Exhaustive scan of a 5×5 grid.

In [51]:
alphas    = [0.001, 0.01, 0.1, 1.0, 10.0]
l1_ratios = [0.0, 0.25, 0.5, 0.75, 1.0]

def grid_search_elasticnet(alphas, l1_ratios, X_tr, y_tr, X_va, y_va):
    best, history = None, []
    for a in alphas:
        for r in l1_ratios:
            m = ElasticNet(alpha=a, l1_ratio=r, random_state=SEED, max_iter=20000).fit(X_tr, y_tr)
            mae = mean_absolute_error(y_va, m.predict(X_va))
            history.append((a, r, mae))
            if best is None or mae < best[2]:
                best = (a, r, mae)
    return best, history

t0 = time.time()
best_grid, hist_grid = grid_search_elasticnet(alphas, l1_ratios, X_tr_s, y_tr, X_va_s, y_va)
t_grid = time.time() - t0
print(f"Grid search: alpha={best_grid[0]}, l1_ratio={best_grid[1]}, MAE={best_grid[2]:.4f}, "
      f"iter={len(hist_grid)}, time={t_grid:.2f}s")


/home/overtkif/school21/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.671e+10, tolerance: 7.308e+06
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/overtkif/school21/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.160e+10, tolerance: 7.308e+06
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklea

Grid search: alpha=1.0, l1_ratio=1.0, MAE=694.7006, iter=25, time=31.43s


/home/overtkif/school21/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.620e+10, tolerance: 7.308e+06
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


### 7.2 Random search
Same parameter budget, but draws random `(alpha, l1_ratio)` samples (log-uniform on `alpha`).

In [52]:
def random_search_elasticnet(n_iter, X_tr, y_tr, X_va, y_va, random_state=SEED):
    rng = np.random.RandomState(random_state)
    best, history = None, []
    for _ in range(n_iter):
        a = 10 ** rng.uniform(-3, 1)
        r = rng.uniform(0.0, 1.0)
        m = ElasticNet(alpha=a, l1_ratio=r, random_state=SEED, max_iter=20000).fit(X_tr, y_tr)
        mae = mean_absolute_error(y_va, m.predict(X_va))
        history.append((a, r, mae))
        if best is None or mae < best[2]:
            best = (a, r, mae)
    return best, history

t0 = time.time()
best_rand, hist_rand = random_search_elasticnet(25, X_tr_s, y_tr, X_va_s, y_va)
t_rand = time.time() - t0
print(f"Random search: alpha={best_rand[0]:.4f}, l1_ratio={best_rand[1]:.4f}, "
      f"MAE={best_rand[2]:.4f}, iter={len(hist_rand)}, time={t_rand:.2f}s")


Random search: alpha=0.0019, l1_ratio=0.8674, MAE=697.2402, iter=25, time=0.56s


### 7.3 Fit best model

In [53]:
best = best_grid if best_grid[2] <= best_rand[2] else best_rand
final_en = ElasticNet(alpha=best[0], l1_ratio=best[1],
                     random_state=SEED, max_iter=20000).fit(X_tr_s, y_tr)
res_en = evaluate(final_en, "ElasticNet best", X_tr_s, y_tr, X_va_s, y_va, X_te_s, y_te)
res_en


{'model': 'ElasticNet best',
 'train': {'MAE': 683.4431066942925,
  'RMSE': 1001.5952102115539,
  'R2': 0.6015127647898747},
 'valid': {'MAE': 694.7006379796769,
  'RMSE': 1016.7261744534917,
  'R2': 0.61284461221094},
 'test': {'MAE': 694.0407196204276,
  'RMSE': 1006.8082613483496,
  'R2': 0.6003832937667299}}

### 7.4 Optuna — Bayesian optimization
Same parameter space, same metric. With a TPE sampler Optuna focuses on promising regions and reaches a comparable optimum with **fewer trials** (20 here vs 25 in the grid).

In [54]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    a = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    r = trial.suggest_float("l1_ratio", 0.0, 1.0)
    m = ElasticNet(alpha=a, l1_ratio=r, random_state=SEED, max_iter=20000).fit(X_tr_s, y_tr)
    return mean_absolute_error(y_va, m.predict(X_va_s))

t0 = time.time()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=20)
t_opt = time.time() - t0
print(f"Optuna: params={study.best_params}, MAE={study.best_value:.4f}, "
      f"trials={len(study.trials)}, time={t_opt:.2f}s")


Optuna: params={'alpha': 0.008605990983240072, 'l1_ratio': 0.9958409935421333}, MAE=695.1986, trials=20, time=0.58s


### 7.5 Compare the three approaches
Lower MAE is better. Optuna typically reaches the same MAE with the same or fewer trials than the grid.

In [55]:
compare = pd.DataFrame([
    {"method": "GridSearch",   "best_MAE": best_grid[2], "iters": len(hist_grid), "seconds": t_grid},
    {"method": "RandomSearch", "best_MAE": best_rand[2], "iters": len(hist_rand), "seconds": t_rand},
    {"method": "Optuna",       "best_MAE": study.best_value, "iters": len(study.trials), "seconds": t_opt},
])
compare


,method,best_MAE,iters,seconds
0,GridSearch,694.700638,25,31.428573
1,RandomSearch,697.240213,25,0.560356
2,Optuna,695.198605,20,0.583251


### 7.6 Optuna inside cross-validation
We combine train+valid and let Optuna optimise the mean K-Fold MAE.

In [56]:
X_full = np.vstack([X_tr_s, X_va_s])
y_full = np.concatenate([y_tr, y_va])
cv_folds = kfold(pd.DataFrame(X_full), 5)

def objective_cv(trial):
    a = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    r = trial.suggest_float("l1_ratio", 0.0, 1.0)
    scores = []
    for tr_idx, te_idx in cv_folds:
        m = ElasticNet(alpha=a, l1_ratio=r, random_state=SEED, max_iter=20000).fit(X_full[tr_idx], y_full[tr_idx])
        scores.append(mean_absolute_error(y_full[te_idx], m.predict(X_full[te_idx])))
    return float(np.mean(scores))

t0 = time.time()
study_cv = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_cv.optimize(objective_cv, n_trials=15)
t_cv = time.time() - t0
print(f"Optuna+CV: params={study_cv.best_params}, mean MAE={study_cv.best_value:.4f}, "
      f"trials={len(study_cv.trials)}, time={t_cv:.2f}s")

# Refit on full train+valid with best CV params, evaluate on the held-out test
final_cv = ElasticNet(**study_cv.best_params, random_state=SEED, max_iter=20000).fit(X_full, y_full)
print("test MAE after Optuna+CV:", mean_absolute_error(y_te, final_cv.predict(X_te_s)))


Optuna+CV: params={'alpha': 0.0018979458542442303, 'l1_ratio': 0.8674044839930883}, mean MAE=691.9760, trials=15, time=2.70s
test MAE after Optuna+CV: 698.7921347934315
